# 🎬 어쩌다지식 - Wan2.1 이미지→영상 변환
정적 AI 일러스트를 살아있는 애니메이션 영상으로 변환합니다.
GPU: A100 권장 (14B 모델), T4에서는 1.3B 모델 자동 선택

In [ ]:
# ==============================
# 설정 (여기만 수정하세요)
# ==============================
RUN_ID = "여기에_run_id_입력"
DRIVE_BASE = "/content/drive/MyDrive/어쩌다지식"
CLIP_DURATION = 5       # 초 (3-5 권장, 길수록 시간/비용 증가)
NUM_FRAMES = 33         # 5초 @ ~6fps (Wan2.1 기본 fps)
# KEY_SCENES_ONLY: True면 훅+클라이맥스 씬만, False면 전체 씬
KEY_SCENES_ONLY = True  # 비용/시간 절감용
KEY_EMOTIONS = ["hook", "surprising", "building", "cta"]  # 이 감정 씬만 생성

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', 
                           '--format=csv,noheader'], capture_output=True, text=True)
gpu_name = gpu_info.stdout.strip()
print(f"GPU: {gpu_name}")

# Select model based on VRAM
if "A100" in gpu_name or "80GB" in gpu_name:
    WAN_MODEL = "Wan-AI/Wan2.1-I2V-14B-480P"
    print("\u2705 14B \ubaa8\ub378 \uc0ac\uc6a9 (\uace0\ud488\uc9c8)")
elif "L4" in gpu_name or "A10" in gpu_name:
    WAN_MODEL = "Wan-AI/Wan2.1-I2V-14B-480P"
    print("\u2705 14B \ubaa8\ub378 \uc0ac\uc6a9")
else:
    WAN_MODEL = "Wan-AI/Wan2.1-I2V-1.3B"
    print("\u26a0\ufe0f  T4 \uac10\uc9c0: 1.3B \ubaa8\ub378 \uc0ac\uc6a9 (\ud488\uc9c8 \ub099\uc74c, Colab Pro \uad8c\uc7a5)")

!pip install -q diffusers transformers accelerate safetensors imageio[ffmpeg] einops

In [ ]:
import json
from pathlib import Path

run_state_path = Path(f"{DRIVE_BASE}/runs/{RUN_ID}.json")
with open(run_state_path) as f:
    state = json.load(f)

script = state["script"]
scenes = script["scenes"]
image_dir = Path(f"{DRIVE_BASE}/runs/{RUN_ID}/images")
video_clip_dir = Path(f"{DRIVE_BASE}/runs/{RUN_ID}/video_clips")
video_clip_dir.mkdir(parents=True, exist_ok=True)

print(f"\u2705 Run: {RUN_ID} | \uc81c\ubaa9: {script['title']}")
print(f"\U0001f5bc\ufe0f  \uc774\ubbf8\uc9c0 \ub514\ub809\ud1a0\ub9ac: {image_dir}")
print(f"\U0001f3ac \ucd9c\ub825 \ub514\ub809\ud1a0\ub9ac: {video_clip_dir}")

# Determine which scenes to process
if KEY_SCENES_ONLY:
    target_scenes = [(i, s) for i, s in enumerate(scenes) 
                     if s.get("emotion", "neutral") in KEY_EMOTIONS]
    print(f"\U0001f3af \ud575\uc2ec \uc528\ub9cc \uccb4\ub9ac: {len(target_scenes)}/{len(scenes)}\uac1c")
else:
    target_scenes = list(enumerate(scenes))
    print(f"\U0001f3ac \uc804\uccb4 \uc528 \uccb4\ub9ac: {len(target_scenes)}\uac1c")

In [ ]:
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import load_image, export_to_video

print(f"\ubaa8\ub378 \ub85c\ub529: {WAN_MODEL}")
pipe = WanImageToVideoPipeline.from_pretrained(
    WAN_MODEL,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
print("\u2705 \ubaa8\ub378 \ub85c\ub4dc \uc644\ub8cc")

In [ ]:
from PIL import Image
import gc

# Prompt templates based on emotion
EMOTION_MOTION_PROMPTS = {
    "hook":       "dramatic camera zoom in, dynamic motion, cinematic movement",
    "surprising": "sudden reveal, fast zoom in, dramatic lighting change",
    "curious":    "slow pan across scene, gentle floating motion",
    "calm":       "slow gentle movement, soft breathing motion, peaceful",
    "building":   "gradual zoom in, rising tension, slow acceleration",
    "warm":       "warm glowing light, gentle sway, soft fade",
    "cta":        "subtle pulse, gentle glow, friendly movement",
    "neutral":    "slow gentle pan, smooth motion",
}

STYLE_SUFFIX = (
    "Korean webtoon illustration style, smooth animation, "
    "high quality, no text overlay"
)

generated = []
for i, scene in target_scenes:
    out_path = video_clip_dir / f"scene_{i:03d}.mp4"
    if out_path.exists():
        print(f"  \uc528 {i}: \uce90\uc2dc \uc0ac\uc6a9")
        generated.append(i)
        continue
    
    image_path = image_dir / f"scene_{i:03d}.jpg"
    if not image_path.exists():
        print(f"  \uc528 {i}: \u26a0\ufe0f \uc774\ubbf8\uc9c0 \uc5c6\uc74c, \uc2a4\ud0b5")
        continue
    
    emotion = scene.get("emotion", "neutral")
    motion_hint = EMOTION_MOTION_PROMPTS.get(emotion, EMOTION_MOTION_PROMPTS["neutral"])
    image_prompt = scene.get("image_prompt", "Korean webtoon illustration")
    prompt = f"{image_prompt}, {motion_hint}, {STYLE_SUFFIX}"
    
    print(f"  \uc528 {i}/{len(scenes)-1}: [{emotion}] \uc601\uc0c1 \uc0dd\uc131 \uc911...")
    
    img = load_image(str(image_path)).resize((832, 480))
    
    with torch.inference_mode():
        frames = pipe(
            image=img,
            prompt=prompt,
            negative_prompt="blurry, low quality, text, watermark, nsfw, ugly",
            num_frames=NUM_FRAMES,
            guidance_scale=5.0,
            num_inference_steps=20,
            generator=torch.Generator().manual_seed(42 + i),
        ).frames[0]
    
    export_to_video(frames, str(out_path), fps=NUM_FRAMES // CLIP_DURATION)
    generated.append(i)
    print(f"    \u2192 {out_path.name} \u2705")
    
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n\u2705 \uc644\ub8cc: {len(generated)}\uac1c \ud074\ub9bd \uc0dd\uc131")

In [ ]:
video_clips = sorted(video_clip_dir.glob("scene_*.mp4"))
state["video_clips"] = [str(p) for p in video_clips]
state["video_clips_key_only"] = KEY_SCENES_ONLY
state.setdefault("stages", {})["wan21_video"] = {
    "status": "completed",
    "clip_count": len(video_clips),
    "key_scenes_only": KEY_SCENES_ONLY,
}

with open(run_state_path, "w") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

print(f"\u2705 {len(video_clips)}\uac1c \uc601\uc0c1 \ud074\ub9bd Drive \uc800\uc7a5 \uc644\ub8cc")
print(f"\n\U0001f680 \ub2e4\uc74c \ub2e8\uacc4 - \ub85c\uce5c\uc5d0\uc11c \uc2e4\ud589:")
print(f"   python pipeline/main.py --skip-to assemble --run-id {RUN_ID}")